In [ ]:
!pip install tellurium

In [ ]:
pip install numpy==1.24.4

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 0.2;    // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;    // Damaged epithelial cells
    species M0 in C1; M0 = 0.07;   // Undifferentiated macrophages
    species M1 in C1; M1 = 0.07;   // M1 macrophages
    species M2 in C1; M2 = 0.07;   // M2 macrophages
    species P in C1;  P = 0.0;     // Pro-inflammatory cytokines
    species A in C1;  A = 0.001;   // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m3 = 0.07;    // Macrophage proliferation rate
    m4 = 0.001;   // Macrophage death rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    K = 1;        // Carrying capacity
    WS = 1;       // Wood smoke exposure (1 = on, 0 = off)

    // Reactions
    // Epithelial cell dynamics
    R1: -> Eh; e1 * (Eh + Ed) * (1 - (Eh + Ed)/K)                 // Proliferation
    R2: Eh -> ; e2 * Eh                                           // Natural death of healthy cells
    R3: Ed -> ; e2 * Ed                                           // Natural death of damaged cells
    R4: Eh -> Ed; e3 * WS * Eh                                    // Damage due to wood smoke
    R5: Ed -> Eh; e4 * Ed                                         // Repair of damaged cells

    // Macrophage dynamics
    R6: M0 -> M1; m1 * (P^nm1 / (Hm1^nm1 + P^nm1)) * M0           // M0 to M1 differentiation
    R7: M0 -> M2; m2 * (A^nm2 / (Hm2^nm2 + A^nm2)) * M0           // M0 to M2 differentiation
    R8: -> M0; m3 * M1 * (1 - (M0 + M1 + M2)/K)                   // M1 proliferation
    R9: -> M0; m3 * M2 * (1 - (M0 + M1 + M2)/K)                   // M2 proliferation
    R10: M0 -> ; m4 * M0                                          // M0 natural death
    R11: M1 -> ; m4 * M1                                          // M1 natural death
    R12: M2 -> ; m4 * M2                                          // M2 natural death
    R13: M1 -> M2; m5 * (A^nm2 / (Hm2^nm2 + A^nm2)) * M1          // M1 to M2 transition

    // Cytokine dynamics
    R14: -> P; e5 * WS * (Eh^ne1 / (He1^ne1 + Eh^ne1)) + e6 * WS * (Ed^ne2 / (He2^ne2 + Ed^ne2)) + m6 * WS * (Hm3^nm3 / (A^nm3 + Hm3^nm3)) * M1     // Pro-inflammatory cytokine production
    R15: P -> ; m7 * P                                            // Pro-inflammatory cytokine degradation
    R16: -> A; m8 * (Hm3^nm3 / (A^nm3 + Hm3^nm3)) * M2 + m9 * (Hm4^nm4 / (A^nm4 + Hm4^nm4)) * M1          // Anti-inflammatory cytokine production
    R17: A -> ; m10 * A                                           // Anti-inflammatory cytokine degradation
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Plot cells
cell_species = ['Eh', 'Ed', 'M0', 'M1', 'M2']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'M0 Macrophages',
               'M1 Macrophages', 'M2 Macrophages']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label)

ax1.set_xlabel('Time')
ax1.set_ylabel('Concentration')
ax1.set_title('Cell Populations')
ax1.legend()

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+6], label=label)  # +6 because P and A are the 6th and 7th columns

ax2.set_xlabel('Time')
ax2.set_ylabel('Concentration')
ax2.set_title('Cytokine Levels')
ax2.legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 0.2;    // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;    // Damaged epithelial cells
    species M0 in C1; M0 = 0.07;   // Undifferentiated macrophages
    species M1 in C1; M1 = 0.07;   // M1 macrophages
    species M2 in C1; M2 = 0.07;   // M2 macrophages
    species P in C1;  P = 0.0;     // Pro-inflammatory cytokines
    species A in C1;  A = 0.001;   // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m3 = 0.07;    // Macrophage proliferation rate
    m4 = 0.001;   // Macrophage death rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    K = 1;        // Carrying capacity
    WS = 0;       // Wood smoke exposure (1 = on, 0 = off)

    // Reactions
    // Epithelial cell dynamics
    R1: -> Eh; e1 * (Eh + Ed) * (1 - (Eh + Ed)/K)                 // Proliferation
    R2: Eh -> ; e2 * Eh                                           // Natural death of healthy cells
    R3: Ed -> ; e2 * Ed                                           // Natural death of damaged cells
    R4: Eh -> Ed; e3 * WS * Eh                                    // Damage due to wood smoke
    R5: Ed -> Eh; e4 * Ed                                         // Repair of damaged cells

    // Macrophage dynamics
    R6: M0 -> M1; m1 * (P^nm1 / (Hm1^nm1 + P^nm1)) * M0           // M0 to M1 differentiation
    R7: M0 -> M2; m2 * (A^nm2 / (Hm2^nm2 + A^nm2)) * M0           // M0 to M2 differentiation
    R8: -> M0; m3 * M1 * (1 - (M0 + M1 + M2)/K)                   // M1 proliferation
    R9: -> M0; m3 * M2 * (1 - (M0 + M1 + M2)/K)                   // M2 proliferation
    R10: M0 -> ; m4 * M0                                          // M0 natural death
    R11: M1 -> ; m4 * M1                                          // M1 natural death
    R12: M2 -> ; m4 * M2                                          // M2 natural death
    R13: M1 -> M2; m5 * (A^nm2 / (Hm2^nm2 + A^nm2)) * M1          // M1 to M2 transition

    // Cytokine dynamics
    R14: -> P; e5 * WS * (Eh^ne1 / (He1^ne1 + Eh^ne1)) + e6 * WS * (Ed^ne2 / (He2^ne2 + Ed^ne2)) + m6 * WS * (Hm3^nm3 / (A^nm3 + Hm3^nm3)) * M1     // Pro-inflammatory cytokine production
    R15: P -> ; m7 * P                                            // Pro-inflammatory cytokine degradation
    R16: -> A; m8 * (Hm3^nm3 / (A^nm3 + Hm3^nm3)) * M2 + m9 * (Hm4^nm4 / (A^nm4 + Hm4^nm4)) * M1          // Anti-inflammatory cytokine production
    R17: A -> ; m10 * A                                           // Anti-inflammatory cytokine degradation
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Plot cells
cell_species = ['Eh', 'Ed', 'M0', 'M1', 'M2']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'M0 Macrophages',
               'M1 Macrophages', 'M2 Macrophages']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label)

ax1.set_xlabel('Time')
ax1.set_ylabel('Concentration')
ax1.set_title('Cell Populations')
ax1.legend()

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+6], label=label)  # +6 because P and A are the 6th and 7th columns

ax2.set_xlabel('Time')
ax2.set_ylabel('Concentration')
ax2.set_title('Cytokine Levels')
ax2.legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
#full model in vivo, with quasi-steady state assumption for differentiation to M1 and M2, a filled epithelium,
#BACKGROUND ANTI-INFLAMMATORY, INCREASED DEATH RATE B/C ADDITION OF EFFLUX
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;    // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;    // Damaged epithelial cells
    species M1 in C1; M1 = 0.07;   // M1 macrophages
    species M2 in C1; M2 = 0.07;   // M2 macrophages
    species P in C1;  P = 0.0;     // Pro-inflammatory cytokines
    species A in C1;  A = 0.001;   // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;   // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1;       // Wood smoke exposure (1 = on, 0 = off)

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*Ed
    M1' = (sM0*m1*(P^nm1 / (Hm1^nm1 + P^nm1)))/(muM0 + m1*(P^nm1 / (Hm1^nm1 + P^nm1)) + m2*(A^nm2 / (Hm2^nm2 + A^nm2))) - m5*(A^nm2 / (Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = (sM0*m2*(A^nm2 / (Hm2^nm2 + A^nm2)))/(muM0 + m1*(P^nm1 / (Hm1^nm1 + P^nm1)) + m2*(A^nm2 / (Hm2^nm2 + A^nm2))) + m5*(A^nm2 / (Hm2^nm2 + A^nm2))*M1 - m4*M2
    P' = e5*WS*(Eh^ne1 / (He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2 / (He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3 / (A^nm3 + Hm3^nm3))*M1 - m7*P
    A' = m8*(Hm3^nm3 / (A^nm3 + Hm3^nm3))*M2 + m9*(Hm4^nm4 / (A^nm4 + Hm4^nm4))*M1 - m10*A + k_c
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Plot cells
cell_species = ['Eh', 'Ed', 'M1', 'M2']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'M1 Macrophages', 'M2 Macrophages']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label)

ax1.set_xlabel('Time')
ax1.set_ylabel('Concentration')
ax1.set_title('Cell Populations')
ax1.legend()

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+5], label=label)  # +5 because P and A are the 5th and 6th columns now

ax2.set_xlabel('Time')
ax2.set_ylabel('Concentration')
ax2.set_title('Cytokine Levels')
ax2.legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
#full model in vivo, with quasi-steady state assumption for differentiation to M1 and M2, a filled epithelium,
#BACKGROUND ANTI-INFLAMMATORY, INCREASED DEATH RATE B/C ADDITION OF EFFLUX
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;    // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;    // Damaged epithelial cells
    species M1 in C1; M1 = 0.07;   // M1 macrophages
    species M2 in C1; M2 = 0.07;   // M2 macrophages
    species P in C1;  P = 0.0;     // Pro-inflammatory cytokines
    species A in C1;  A = 0.001;   // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;   // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1;       // Wood smoke exposure (1 = on, 0 = off)

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*Ed



    M1' = (sM0*m1*(P^nm1 / (Hm1^nm1 + P^nm1)))/(muM0 + m1*(P^nm1 / (Hm1^nm1 + P^nm1)) + m2*(A^nm2 / (Hm2^nm2 + A^nm2))) - m5*(A^nm2 / (Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = (sM0*m2*(A^nm2 / (Hm2^nm2 + A^nm2)))/(muM0 + m1*(P^nm1 / (Hm1^nm1 + P^nm1)) + m2*(A^nm2 / (Hm2^nm2 + A^nm2))) + m5*(A^nm2 / (Hm2^nm2 + A^nm2))*M1 - m4*M2
    P' = e5*WS*(Eh^ne1 / (He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2 / (He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3 / (A^nm3 + Hm3^nm3))*M1 - m7*P
    A' = m8*(Hm3^nm3 / (A^nm3 + Hm3^nm3))*M2 + m9*(Hm4^nm4 / (A^nm4 + Hm4^nm4))*M1 - m10*A + k_c
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
sns.set_style("whitegrid")
sns.set_palette("colorblind")

# Plot cells
cell_species = ['Eh', 'Ed', 'M1', 'M2']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'M1 Macrophages', 'M2 Macrophages']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label)

ax1.set_xlabel('Time')
ax1.set_ylabel('Concentration')
ax1.set_title('Cell Populations')
ax1.legend()

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+5], label=label)  # +5 because P and A are the 5th and 6th columns now

ax2.set_xlabel('Time')
ax2.set_ylabel('Concentration')
ax2.set_title('Cytokine Levels')
ax2.legend()

# Adjust layout and show the plot
plt.tight_layout()
plt.show()

In [ ]:
#full model in vivo, added resident macrophage population with wood smoke dependent proliferation plus wood smoke dependent influx
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    sM0 = 0.1;    // Influx rate of M0 macrophages
    muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate

    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke dosing event
  # E1: at (1 < time < 2): WS = 1.0;
  # E2: at (time >= 2): WS = 0.0;

  # // Wood smoke continuous exposure
  # E1: at (time > 1): WS = 1.0;



  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 12))
sns.set_style("white")
sns.set_palette("colorblind")
line_styles = ['-', '--', '-.', ':']

# Plot cells
cell_species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
               'Resident M0', 'Resident M1', 'Resident M2']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)
ax1.set_xlim(0.0,200)
ax1.legend(fontsize=12)
ax1.grid(False)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+8], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax2.set_xlabel('Time', fontsize=14)
ax2.set_ylabel('Concentration', fontsize=14)
ax2.set_xlim(0.0,200)
ax2.legend(fontsize=12)
ax2.grid(False)

# Customize tick label sizes
ax2.tick_params(axis='both', which='major', labelsize=12)

# Adjust layout and show the plot
plt.tight_layout()


# plt.savefig('full model_one_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_one_exp.eps', format='eps', bbox_inches='tight')

# plt.savefig('full model_cont_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_cont_exp.eps', format='eps', bbox_inches='tight')

plt.savefig('full model_rep_exp.png', dpi=300, bbox_inches='tight')
plt.savefig('full model_rep_exp.eps', format='eps', bbox_inches='tight')
plt.show()

In [ ]:
#full model in vivo, added resident macrophage population with wood smoke dependent proliferation plus wood smoke dependent influx
#adding modulation of repair processes by M2 and A
import tellurium as te
import matplotlib.pyplot as plt
import seaborn as sns

# Define the full model
model = '''
    // Compartments
    compartment C1;

    // Species
    species Eh in C1; Eh = 1.0;     // Healthy epithelial cells
    species Ed in C1; Ed = 0.0;     // Damaged epithelial cells
    species M1 in C1; M1 = 0.001;    // Recruited M1 macrophages
    species M2 in C1; M2 = 0.001;    // Recruited M2 macrophages
    species M0R in C1; M0R = 0.0;  // Resident M0 macrophages
    species M1R in C1; M1R = 0.0;   // Resident M1 macrophages
    species M2R in C1; M2R = 0.001;   // Resident M2 macrophages, low M2 dominant population
    species P in C1; P = 0.0;       // Pro-inflammatory cytokines
    species A in C1; A = 0.000;     // Anti-inflammatory cytokines

    // Parameters
    e1 = 0.0315;  // Epithelial cell proliferation rate
    e2 = 0.0001;   // Epithelial cell death rate
    e3 = 0.005;   // Healthy epithelial cell damage rate
    e4 = 0.002;   // Damaged epithelial cell repair rate
    e5 = 0.1;     // Healthy epithelial pro-inflammatory cytokine production rate
    e6 = 0.2;     // Damaged epithelial pro-inflammatory cytokine production rate
    He1 = 0.2;    // Hill constant for P production by Eh
    He2 = 0.2;    // Hill constant for P production by Ed
    ne1 = 4;      // Hill coefficient for P production by Eh
    ne2 = 4;      // Hill coefficient for P production by Ed

    m1 = 50;      // M0 to M1 differentiation rate
    m2 = 0.1;     // M0 to M2 differentiation rate
    m4 = 0.02;    // Macrophage decay (death/efflux) rate
    m5 = 0.05;    // M1 to M2 transition rate
    m6 = 0.2;     // M1 pro-inflammatory cytokine production rate
    m7 = 0.04;    // Pro-inflammatory cytokine degradation rate
    m8 = 0.1;     // M2 anti-inflammatory cytokine production rate
    m9 = 0.04;    // M1 anti-inflammatory cytokine production rate
    m10 = 0.04;   // Anti-inflammatory cytokine degradation rate
    Hm1 = 0.25;   // Hill constant for P activation
    Hm2 = 0.05;   // Hill constant for A activation
    Hm3 = 0.9;    // Hill constant for A inhibition
    Hm4 = 1.2;    // Hill constant for A self-inhibition
    nm1 = 2.5;    // Hill coefficient for P activation
    nm2 = 4;      // Hill coefficient for A activation
    nm3 = 2;      // Hill coefficient for A inhibition
    nm4 = 2;      // Hill coefficient for A self-inhibition

    # sM0 = 0.1;    // Influx rate of M0 macrophages
    # muM0 = 0.05;  // Efflux/death rate of M0 macrophages
    k_c = 0.0125; // Concentration of background anti-inflammatory cytokine

    K = 1;        // Carrying capacity
    WS = 1.0;       // Wood smoke exposure (1 = on, 0 = off)
    m11 = 0.2;    // Direct WS activation rate for M1
    m3 = 0.07;    // Resident macrophage proliferation rate
    smr = 0.1;    // Circulating macrophage source rate
    mumr = 0.05;  // Circulating macrophage decay rate
    e10 = 0.0;   // Baseline A production by healthy epithelial cells
    e8 = 1.0;     // M2 enhancement of repair
    e9 = 1.0;    // A enhancement of repair


    // Helper functions for activation/influx
    RM1 := m1*(P^nm1/(Hm1^nm1 + P^nm1)) + m11*WS    // M1 activation function
    RM2 := m2*(A^nm2/(Hm2^nm2 + A^nm2)) + k_c       // M2 activation function

    // Differential equations
    Eh' = e1*(Eh + Ed)*(1 - (Eh + Ed)/K) - e2*Eh - e3*WS*Eh + e4*(1 + e9*(M2 + M2R) + e10*A)*Ed
    Ed' = -e2*Ed + e3*WS*Eh - e4*(1 + e8*(M2 + M2R) + e9*A)*Ed

    // Recruited macrophages
    M1' = WS*(smr*RM1)/(mumr + RM1 + RM2) - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M1
    M2' = WS*(smr*RM2)/(mumr + RM1 + RM2) + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1 - m4*M2

    // Resident macrophages
    M0R' = -m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R  - m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m3*WS*(M1R + M2R)*(1 - (M0R + M1R + M2R)/K) - m4*M0R
    M1R' = m1*(P^nm1/(Hm1^nm1 + P^nm1))*M0R - m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M1R
    M2R' = m2*(A^nm2/(Hm2^nm2 + A^nm2))*M0R + m5*(A^nm2/(Hm2^nm2 + A^nm2))*M1R - m4*M2R

    // Cytokine equations including both resident and recruited macrophages
    P' = e5*WS*(Eh^ne1/(He1^ne1 + Eh^ne1)) + e6*WS*(Ed^ne2/(He2^ne2 + Ed^ne2)) + m6*WS*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M1 + M1R) - m7*P
    A' = m8*(Hm3^nm3/(A^nm3 + Hm3^nm3))*(M2 + M2R) + m9*(Hm4^nm4/(A^nm4 + Hm4^nm4))*(M1 + M1R) - m10*A + k_c

  # // Wood smoke dosing event
  # E1: at (1 < time < 2): WS = 1.0;
  # E2: at (time >= 2): WS = 0.0;

  # // Wood smoke continuous exposure
  # E1: at (time > 1): WS = 1.0;



  // Wood smoke recurring dosing events for 3 meals a day
  E1: at (time % 24 >= 8 and time % 24 < 9): WS = 1.0;
  E2: at (time % 24 >= 9 and time % 24 < 13): WS = 0.0;
  E3: at (time % 24 >= 13 and time % 24 < 14): WS = 1.0;
  E4: at (time % 24 >= 14 and time % 24 < 18): WS = 0.0;
  E5: at (time % 24 >= 18 and time % 24 < 19): WS = 1.0;
  E6: at (time % 24 >= 19 and time % 24 < 24): WS = 0.0;
'''

# Load the model
r = te.loada(model)

# Run simulation
result = r.simulate(0, 200, 1000)

# Set up the plots
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
sns.set_style("white")
sns.set_palette("colorblind")
line_styles = ['-', '--', '-.', ':']

# Plot cells
cell_species = ['Eh', 'Ed', 'M1', 'M2', 'M0R', 'M1R', 'M2R']
cell_labels = ['Healthy Epithelial', 'Damaged Epithelial', 'Recruited M1', 'Recruited M2',
               'Resident M0', 'Resident M1', 'Resident M2']

for i, (species_name, label) in enumerate(zip(cell_species, cell_labels)):
    ax1.plot(result[:, 0], result[:, i+1], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax1.set_xlabel('Time', fontsize=14)
ax1.set_ylabel('Concentration', fontsize=14)
ax1.set_xlim(0.0,200)
ax1.legend(fontsize=12)
ax1.grid(False)

# Customize tick label sizes
ax1.tick_params(axis='both', which='major', labelsize=12)

# Plot cytokines
cytokine_species = ['P', 'A']
cytokine_labels = ['Pro-inflammatory Cytokines', 'Anti-inflammatory Cytokines']

for i, (species_name, label) in enumerate(zip(cytokine_species, cytokine_labels)):
    ax2.plot(result[:, 0], result[:, i+8], label=label,
             linestyle=line_styles[i % len(line_styles)])

ax2.set_xlabel('Time', fontsize=14)
ax2.set_ylabel('Concentration', fontsize=14)
ax2.set_xlim(0.0,200)
ax2.legend(fontsize=12)
ax2.grid(False)

# Customize tick label sizes
ax2.tick_params(axis='both', which='major', labelsize=12)

# Adjust layout and show the plot
plt.tight_layout()


# plt.savefig('full model_one_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_one_exp.eps', format='eps', bbox_inches='tight')

# plt.savefig('full model_cont_exp.png', dpi=300, bbox_inches='tight')
# plt.savefig('full model_cont_exp.eps', format='eps', bbox_inches='tight')

plt.savefig('full model_rep_exp.png', dpi=300, bbox_inches='tight')
plt.savefig('full model_rep_exp.eps', format='eps', bbox_inches='tight')
plt.show()